Notebook 04 - Consulta RAG da Base Maicon com LangChain e OpenRouter

Este notebook implementa a etapa de consulta do sistema RAG utilizando a Base Maicon.

O índice FAISS e os metadados gerados no Notebook 03 são carregados para realizar a recuperação semântica dos chunks mais relevantes. A pergunta do usuário é transformada em embedding utilizando o mesmo modelo empregado na indexação da base.

Os embeddings da base foram gerados a partir de uma estratégia de prioridade textual definida no Notebook 02: `cleaned_summary → summary → text`. O campo `embedding_source` preserva a origem utilizada na representação vetorial de cada chunk, permitindo a rastreabilidade dos resultados recuperados.

Os chunks recuperados pelo FAISS são utilizados para construir o contexto enviado ao modelo de linguagem. O texto original de cada chunk é preservado nos metadados e utilizado na construção desse contexto.

A geração da resposta é realizada por meio do OpenRouter, utilizando o modelo Llama 3.1 8B Instruct. O sistema seleciona uma alternativa entre A e E, apresenta uma justificativa baseada exclusivamente no contexto recuperado e registra os principais documentos utilizados.

Ao final de cada consulta, os detalhes da recuperação e da resposta são salvos automaticamente na pasta `04_Resultados`, permitindo a posterior análise estatística e avaliação do sistema.

Preparação do Ambiente

In [ ]:
# Instala as bibliotecas necessárias para o Notebook 04

!pip install -q \
    faiss-cpu \
    sentence-transformers \
    langchain \
    langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 26.9 MB/s eta 0:00:00


In [ ]:
# Importa as bibliotecas necessárias

import json
import re

from datetime import datetime
from getpass import getpass
from pathlib import Path

import faiss
import numpy as np

from google.colab import drive

from sentence_transformers import SentenceTransformer

from langchain_openai import ChatOpenAI

In [ ]:
# Monta o Google Drive

drive.mount(
    "/content/drive"
)

Mounted at /content/drive


Funções Auxiliares

In [ ]:
# Define funções auxiliares para padronizar as mensagens

def print_header(title):
    print("\n" + "=" * 70)
    print(f" {title}")
    print("=" * 70)


def print_success(message):
    print(f"\n✅ {message}")


def print_error(message):
    print(f"\n❌ {message}")


def print_warning(message):
    print(f"\n⚠️ {message}")


def print_info(label, value):
    print(f"{label:<25} {value}")

Configuração do Projeto

In [ ]:
# Define os caminhos utilizados no Notebook 04

PROJECT_PATH = Path(
    "/content/drive/MyDrive/RAG_Novo"
)

FAISS_DIR = (
    PROJECT_PATH
    / "03_FAISS"
)

RESULTS_DIR = (
    PROJECT_PATH
    / "04_Resultados"
)

FAISS_INDEX_FILE = (
    FAISS_DIR
    / "faiss.index"
)

FAISS_METADATA_FILE = (
    FAISS_DIR
    / "faiss_metadata.json"
)

FAISS_INFO_FILE = (
    FAISS_DIR
    / "index_info.json"
)

print_header(
    "CONFIGURAÇÃO DO PROJETO"
)

print_info(
    "Projeto:",
    PROJECT_PATH
)

print_info(
    "FAISS:",
    FAISS_DIR
)

print_info(
    "Resultados:",
    RESULTS_DIR
)

print_info(
    "Índice FAISS:",
    FAISS_INDEX_FILE
)

print_info(
    "Metadados:",
    FAISS_METADATA_FILE
)

print_info(
    "Manifesto:",
    FAISS_INFO_FILE
)

print_success(
    "Caminhos configurados com sucesso."
)

print("=" * 70)


 CONFIGURAÇÃO DO PROJETO
Projeto:                  /content/drive/MyDrive/RAG_Novo
FAISS:                    /content/drive/MyDrive/RAG_Novo/03_FAISS
Resultados:               /content/drive/MyDrive/RAG_Novo/04_Resultados
Índice FAISS:             /content/drive/MyDrive/RAG_Novo/03_FAISS/faiss.index
Metadados:                /content/drive/MyDrive/RAG_Novo/03_FAISS/faiss_metadata.json
Manifesto:                /content/drive/MyDrive/RAG_Novo/03_FAISS/index_info.json

✅ Caminhos configurados com sucesso.


Verificação dos Artefatos

In [ ]:
# Verifica se todos os artefatos necessários existem

print_header(
    "VERIFICAÇÃO DOS ARTEFATOS"
)

required_files = {

    "Índice FAISS": FAISS_INDEX_FILE,

    "Metadados FAISS": FAISS_METADATA_FILE,

    "Manifesto": FAISS_INFO_FILE

}

missing_files = []

for name, path in required_files.items():

    if path.exists():

        print_info(
            f"{name}:",
            "OK"
        )

    else:

        print_error(
            f"{name} não encontrado."
        )

        print_info(
            "Esperado em:",
            path
        )

        missing_files.append(
            str(path)
        )

if missing_files:

    raise FileNotFoundError(
        "Existem artefatos obrigatórios ausentes."
    )

print_success(
    "Todos os artefatos foram localizados."
)

print("=" * 70)


 VERIFICAÇÃO DOS ARTEFATOS
Índice FAISS:             OK
Metadados FAISS:          OK
Manifesto:                OK

✅ Todos os artefatos foram localizados.


Carregamento do Índice Metadados

In [ ]:
# Carrega o índice FAISS, os metadados e o manifesto

print_header(
    "CARREGAMENTO DOS ARTEFATOS"
)

# Índice FAISS
index = faiss.read_index(
    str(FAISS_INDEX_FILE)
)

# Metadados
with open(
    FAISS_METADATA_FILE,
    "r",
    encoding="utf-8"
) as file:

    metadata_records = json.load(
        file
    )

# Manifesto
with open(
    FAISS_INFO_FILE,
    "r",
    encoding="utf-8"
) as file:

    index_info = json.load(
        file
    )

print_info(
    "Vetores no índice:",
    index.ntotal
)

print_info(
    "Metadados:",
    len(
        metadata_records
    )
)

print_info(
    "Tipo do índice:",
    index_info[
        "index_type"
    ]
)

print_info(
    "Dimensão:",
    index_info[
        "embedding_dimension"
    ]
)

print_success(
    "Artefatos carregados com sucesso."
)

print("=" * 70)


 CARREGAMENTO DOS ARTEFATOS
Vetores no índice:        6844
Metadados:                6844
Tipo do índice:           IndexFlatIP
Dimensão:                 768

✅ Artefatos carregados com sucesso.


Validação dos Artefatos

In [ ]:
# Valida a consistência dos artefatos carregados

print_header(
    "VALIDAÇÃO DOS ARTEFATOS"
)

print_info(
    "Vetores no índice:",
    index.ntotal
)

print_info(
    "Metadados:",
    len(metadata_records)
)

print_info(
    "Tipo do índice:",
    index_info["index_type"]
)

print_info(
    "Dimensão:",
    index_info["embedding_dimension"]
)

print_info(
    "Modelo de embeddings:",
    index_info["embedding_model"]
)

print()

# Recupera as contagens de origem dos embeddings
source_counts = index_info.get(
    "embedding_source_counts",
    {}
)

print_info(
    "Fonte cleaned_summary:",
    source_counts.get(
        "cleaned_summary",
        0
    )
)

print_info(
    "Fonte summary:",
    source_counts.get(
        "summary",
        0
    )
)

print_info(
    "Fonte text:",
    source_counts.get(
        "text",
        0
    )
)

# Valida quantidade de vetores e metadados
if index.ntotal != len(
    metadata_records
):

    raise ValueError(
        "O número de vetores no índice não corresponde "
        "ao número de metadados."
    )

# Valida dimensão
if index_info["embedding_dimension"] != 768:

    raise ValueError(
        "A dimensão registrada no manifesto está incorreta."
    )

# Valida tipo do índice
if index_info["index_type"] != "IndexFlatIP":

    raise ValueError(
        "O tipo do índice não é o esperado."
    )

# Valida total registrado no manifesto
if index_info.get(
    "total_vectors"
) != index.ntotal:

    raise ValueError(
        "O total de vetores registrado no manifesto "
        "não corresponde ao índice carregado."
    )

# Valida as origens dos embeddings
if sum(
    source_counts.values()
) != len(
    metadata_records
):

    raise ValueError(
        "A contagem das fontes dos embeddings "
        "não corresponde ao total de metadados."
    )

# Confere se todos os metadados possuem embedding_source válido
valid_sources = {
    "cleaned_summary",
    "summary",
    "text"
}

invalid_source_records = [
    record
    for record in metadata_records
    if record.get(
        "embedding_source"
    ) not in valid_sources
]

if invalid_source_records:

    raise ValueError(
        f"Foram encontrados "
        f"{len(invalid_source_records)} registros "
        "com embedding_source inválido."
    )

print_success(
    "Artefatos e fontes dos embeddings validados com sucesso."
)

print("=" * 70)


 VALIDAÇÃO DOS ARTEFATOS
Vetores no índice:        6844
Metadados:                6844
Tipo do índice:           IndexFlatIP
Dimensão:                 768
Modelo de embeddings:     sentence-transformers/paraphrase-multilingual-mpnet-base-v2

Fonte cleaned_summary:    6679
Fonte summary:            7
Fonte text:               158

✅ Artefatos e fontes dos embeddings validados com sucesso.


Carregamento do Modelo de Embeddings

In [ ]:
# Carrega o mesmo modelo de embeddings utilizado na indexação

print_header(
    "CARREGAMENTO DO MODELO DE EMBEDDINGS"
)

EMBEDDING_MODEL_NAME = (
    index_info[
        "embedding_model"
    ]
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

embedding_dimension = (
    embedding_model
    .get_embedding_dimension()
)

print_info(
    "Modelo:",
    EMBEDDING_MODEL_NAME
)

print_info(
    "Dimensão:",
    embedding_dimension
)

if embedding_dimension != index_info["embedding_dimension"]:

    raise ValueError(
        "A dimensão do modelo de embeddings "
        "não corresponde à dimensão do índice FAISS."
    )

print_success(
    "Modelo de embeddings carregado com sucesso."
)

print("=" * 70)


 CARREGAMENTO DO MODELO DE EMBEDDINGS


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo:                   sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Dimensão:                 768

✅ Modelo de embeddings carregado com sucesso.


Configuração da Recuperação

Escolha do parâmetro `TOP_K`

Após a geração do embedding da pergunta, o sistema realiza uma busca no índice FAISS para recuperar os trechos mais semelhantes à consulta. A quantidade de trechos recuperados é definida pelo parâmetro **`TOP_K`**, que representa o número de chunks utilizados para compor o contexto enviado ao modelo de linguagem.

Nesta implementação foi adotado **`TOP_K = 3`**, ou seja, para cada pergunta são recuperados os **três trechos mais relevantes** da base de conhecimento.

A definição desse valor busca equilibrar a quantidade de informações fornecidas ao modelo e a relevância do contexto recuperado:

- **Valores muito baixos** (por exemplo, `TOP_K = 1`) podem não fornecer contexto suficiente para responder adequadamente a perguntas que dependem de informações distribuídas em mais de um trecho.
- **Valores muito altos** (por exemplo, `TOP_K = 10`) podem recuperar conteúdos menos relevantes, aumentar o tamanho do contexto enviado ao modelo e introduzir informações desnecessárias, o que pode prejudicar a qualidade da resposta.
- **`TOP_K = 3`** representa um equilíbrio entre cobertura e precisão, fornecendo contexto suficiente para a maioria das consultas sem aumentar excessivamente a quantidade de texto enviada ao modelo de linguagem.

O valor de `TOP_K` não é fixo e pode ser ajustado de acordo com as características da base documental, a complexidade das perguntas e os resultados obtidos durante a avaliação do sistema.

In [ ]:
# Configuração da recuperação semântica

print_header(
    "CONFIGURAÇÃO DA RECUPERAÇÃO"
)

TOP_K = 3

print_info(
    "Top-K:",
    TOP_K
)

print_success(
    "Recuperação configurada com sucesso."
)

print("=" * 70)


 CONFIGURAÇÃO DA RECUPERAÇÃO
Top-K:                    3

✅ Recuperação configurada com sucesso.


Função de Recuperação Semântica

In [ ]:
# Recupera os chunks mais relevantes para uma pergunta

def retrieve_chunks(
    question: str,
    top_k: int = TOP_K
):

    if not question.strip():

        raise ValueError(
            "A pergunta não pode estar vazia."
        )

    # Gera o embedding da pergunta
    query_embedding = embedding_model.encode(
        question,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(
        np.float32
    )

    # Consulta o índice FAISS
    scores, indices = index.search(
        query_embedding.reshape(1, -1),
        top_k
    )

    retrieved_chunks = []

    for idx, score in zip(
        indices[0],
        scores[0]
    ):

        if idx < 0:
            continue

        chunk = metadata_records[
            idx
        ].copy()

        chunk["score"] = float(
            score
        )

        retrieved_chunks.append(
            chunk
        )

    return retrieved_chunks

Construção do Contexto

In [ ]:
# Constrói o contexto a partir dos chunks recuperados

def build_context(
    retrieved_chunks
):

    context_parts = []

    for i, chunk in enumerate(
        retrieved_chunks,
        start=1
    ):

        context_parts.append(
            f"""Document {i}

Article: {chunk['article_name']}
Chunk: {chunk['chunk_id']}

{chunk['original_text']}
"""
        )

    context = "\n\n".join(
        context_parts
    )

    return context

Construção do Prompt

In [ ]:
# Define a instrução e o formato do prompt do RAG

instruction = """
You are an AI assistant specialized in Technology and Innovation Roadmapping.

Your task is to answer multiple-choice questions using ONLY the information
contained in the retrieved context.

Carefully compare all alternatives A, B, C, D and E with the retrieved context
before selecting the answer.

Select exactly one alternative only when its complete meaning is supported
by the retrieved documents.

Pay close attention to:
- the exact terminology used in the documents;
- the order of concepts when the question asks for a sequence;
- distinctions between similar concepts;
- whether every element of an alternative is actually supported by the context.

Do not select an alternative if it contains a term, relationship or ordering
that is contradicted by or absent from the relevant context.

The value of "Answer" MUST reproduce exactly the full text of the selected
alternative as written in the question.

Provide a concise justification based exclusively on the retrieved documents.

Do not invent information or use external knowledge.

If the retrieved context does not contain enough information to determine
the correct alternative, state this clearly instead of guessing.

Identify the documents and chunks that directly support the selected answer.
Do not cite a document unless it actually supports the answer.

Return ONLY a valid JSON object using exactly this structure:

{
"Correct Alternative": "A, B, C, D or E",
"Answer": "Exact full text of the selected alternative",
"Justification": "Brief explanation based on the retrieved context",
"Sources": [
{
"Article": "article name",
"Chunk": "chunk number"
}
]
}

Do NOT include markdown.
Do NOT include text before or after the JSON.
Do NOT explain your reasoning process.
Return only the final JSON object.
""".strip()

rag_prompt_template = """### Instruction:

{}

### Context:

{}

### Input:

{}

### Response:
"""

Configuração do OpenRouter

In [ ]:
# Configuração do modelo LLM utilizado pelo RAG

LLM_MODEL_NAME = "meta-llama/llama-3.1-8b-instruct"

print_header(
    "CONFIGURAÇÃO DO MODELO LLM"
)

print_info(
    "Provedor:",
    "OpenRouter"
)

print_info(
    "Modelo:",
    LLM_MODEL_NAME
)

print_success(
    "Modelo configurado com sucesso."
)

print("=" * 70)


 CONFIGURAÇÃO DO MODELO LLM
Provedor:                 OpenRouter
Modelo:                   meta-llama/llama-3.1-8b-instruct

✅ Modelo configurado com sucesso.


Configuração da API

A comunicação com o modelo de linguagem é realizada por meio da API do OpenRouter.

Por questões de segurança, a chave de acesso não é armazenada no notebook. Ela é informada durante a execução utilizando a função `getpass()`, que permite inserir a chave de forma oculta, evitando sua exposição no código ou nos arquivos compartilhados.

In [ ]:
# Configuração da API do OpenRouter

print_header(
    "CONFIGURAÇÃO DA API"
)

OPENROUTER_API_KEY = getpass(
    "Digite sua API Key do OpenRouter: "
)

if not OPENROUTER_API_KEY:

    raise ValueError(
        "A chave da API não foi informada."
    )

print_info(
    "Status da API:",
    "Configurada"
)

print_success(
    "API do OpenRouter configurada com sucesso."
)

print("=" * 70)


 CONFIGURAÇÃO DA API
Digite sua API Key do OpenRouter: ··········
Status da API:            Configurada

✅ API do OpenRouter configurada com sucesso.


Configuração do Cliente OpenRouter

In [ ]:
# Configura o cliente do OpenRouter via LangChain

print_header(
    "CONFIGURAÇÃO DO CLIENTE OPENROUTER"
)

if not OPENROUTER_API_KEY:

    raise ValueError(
        "A chave OPENROUTER_API_KEY não foi configurada."
    )

llm = ChatOpenAI(
    model=LLM_MODEL_NAME,
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

print_info(
    "Modelo:",
    LLM_MODEL_NAME
)

print_info(
    "Temperatura:",
    "Padrão do modelo/provedor (não definida manualmente)"
)

print_success(
    "Cliente OpenRouter configurado com sucesso."
)

print("=" * 70)


 CONFIGURAÇÃO DO CLIENTE OPENROUTER
Modelo:                   meta-llama/llama-3.1-8b-instruct
Temperatura:              Padrão do modelo/provedor (não definida manualmente)

✅ Cliente OpenRouter configurado com sucesso.


Construção do Prompt Completo

In [ ]:
# Constrói o prompt completo enviado ao modelo

def build_prompt(
    question: str,
    retrieved_chunks
):

    context = build_context(
        retrieved_chunks
    )

    prompt = rag_prompt_template.format(
        instruction,
        context,
        question
    )

    return prompt

Validação da Resposta do LLM

In [ ]:
# Valida e padroniza a resposta estruturada retornada pelo LLM
# Também recupera os campos quando o JSON retornado está levemente malformado.

def validate_llm_answer(
    parsed_answer,
    raw_answer
):

    # --------------------------------------------------
    # GARANTE QUE raw_answer SEJA TEXTO
    # --------------------------------------------------

    if raw_answer is None:
        raw_answer = ""

    raw_answer = str(
        raw_answer
    ).strip()

    # --------------------------------------------------
    # ESTRUTURA INICIAL
    # --------------------------------------------------

    if isinstance(
        parsed_answer,
        dict
    ):

        working_answer = dict(
            parsed_answer
        )

    else:

        working_answer = {}

    # --------------------------------------------------
    # RECUPERA A ALTERNATIVA
    # --------------------------------------------------

    correct_alternative = str(
        working_answer.get(
            "Correct Alternative",
            ""
        )
    ).strip().upper()

    # Se não veio corretamente no JSON,
    # tenta recuperar diretamente da resposta bruta.
    if correct_alternative not in {
        "A", "B", "C", "D", "E"
    }:

        alternative_match = re.search(
            r'"Correct Alternative"\s*:\s*"([A-E])"',
            raw_answer,
            flags=re.IGNORECASE
        )

        if alternative_match:

            correct_alternative = (
                alternative_match
                .group(1)
                .upper()
            )

        else:

            correct_alternative = ""

    # --------------------------------------------------
    # RECUPERA A RESPOSTA
    # --------------------------------------------------

    answer = working_answer.get(
        "Answer",
        ""
    )

    if answer is None:
        answer = ""

    answer = str(
        answer
    ).strip()

    # Se não foi recuperada pelo JSON,
    # tenta extrair da resposta bruta.
    if not answer:

        answer_match = re.search(
            r'"Answer"\s*:\s*"([^"]*)"',
            raw_answer,
            flags=re.IGNORECASE | re.DOTALL
        )

        if answer_match:

            answer = (
                answer_match
                .group(1)
                .strip()
            )

    # --------------------------------------------------
    # RECUPERA A JUSTIFICATIVA
    # --------------------------------------------------

    justification = working_answer.get(
        "Justification",
        ""
    )

    if justification is None:
        justification = ""

    justification = str(
        justification
    ).strip()

    # Em alguns erros de parsing anteriores,
    # raw_answer inteiro pode ter sido colocado
    # indevidamente em Justification.
    if (
        not justification
        or justification == raw_answer
        or '"Correct Alternative"' in justification
    ):

        justification = ""

        # Caso normal: justificativa entre aspas
        justification_match = re.search(
            r'"Justification"\s*:\s*"(.+?)"\s*,\s*"Sources"',
            raw_answer,
            flags=re.IGNORECASE | re.DOTALL
        )

        if justification_match:

            justification = (
                justification_match
                .group(1)
                .strip()
            )

        else:

            # Caso observado no experimento:
            # o modelo esquece as aspas em volta
            # da justificativa.
            justification_match = re.search(
                r'"Justification"\s*:\s*(.+?)\s*,\s*"Sources"\s*:',
                raw_answer,
                flags=re.IGNORECASE | re.DOTALL
            )

            if justification_match:

                justification = (
                    justification_match
                    .group(1)
                    .strip()
                    .strip('"')
                )

    # --------------------------------------------------
    # RECUPERA AS FONTES
    # --------------------------------------------------

    sources = working_answer.get(
        "Sources",
        []
    )

    if not isinstance(
        sources,
        list
    ):

        sources = []

    validated_sources = []

    # Primeiro utiliza fontes já interpretadas,
    # caso o JSON tenha sido válido.
    for source in sources:

        if not isinstance(
            source,
            dict
        ):
            continue

        article = str(
            source.get(
                "Article",
                ""
            )
        ).strip()

        chunk = str(
            source.get(
                "Chunk",
                ""
            )
        ).strip()

        if article or chunk:

            validated_sources.append(
                {
                    "Article": article,
                    "Chunk": chunk
                }
            )

    # Se nenhuma fonte foi interpretada,
    # tenta recuperar diretamente do texto bruto.
    if not validated_sources:

        source_matches = re.findall(
            r'"Article"\s*:\s*"([^"]*)"\s*,\s*'
            r'"Chunk"\s*:\s*"([^"]*)"',
            raw_answer,
            flags=re.IGNORECASE | re.DOTALL
        )

        for article, chunk in source_matches:

            validated_sources.append(
                {
                    "Article": article.strip(),
                    "Chunk": chunk.strip()
                }
            )

    # --------------------------------------------------
    # RETORNO PADRONIZADO
    # --------------------------------------------------

    return {
        "Correct Alternative": correct_alternative,
        "Answer": answer,
        "Justification": justification,
        "Sources": validated_sources
    }

Consulta Completa do RAG

In [ ]:
# Executa o fluxo completo do sistema RAG e interpreta a resposta JSON retornada pelo LLM

def rag_query(
    question: str,
    top_k: int = TOP_K
):

    # Validação da pergunta
    if not isinstance(question, str) or not question.strip():

        raise ValueError(
            "A pergunta não pode estar vazia."
        )

    # Validação do número de documentos recuperados
    if not isinstance(top_k, int) or top_k <= 0:

        raise ValueError(
            "TOP_K deve ser um número inteiro maior que zero."
        )

    # Verifica se o cliente LLM foi inicializado
    if llm is None:

        raise RuntimeError(
            "O cliente OpenRouter não foi inicializado."
        )

    # Recupera os chunks mais relevantes
    retrieved_chunks = retrieve_chunks(
        question=question,
        top_k=top_k
    )

    if not retrieved_chunks:

        raise RuntimeError(
            "Nenhum trecho relevante foi recuperado pelo FAISS."
        )

    # Constrói o prompt com o contexto recuperado
    full_prompt = build_prompt(
        question=question,
        retrieved_chunks=retrieved_chunks
    )

    # Envia o prompt ao Llama via OpenRouter
    try:

        response = llm.invoke(
            full_prompt
        )

    except Exception as error:

        raise RuntimeError(
            "Não foi possível obter uma resposta do modelo "
            "Llama 3.1 8B Instruct via OpenRouter. "
            "Verifique a conexão, a API Key, os créditos da conta "
            "e a disponibilidade do modelo."
        ) from error

    # Extrai a resposta textual
    raw_answer = str(
        response.content
    ).strip()

    if not raw_answer:

        raise RuntimeError(
            "O modelo retornou uma resposta vazia."
        )

    # Tenta interpretar a resposta como JSON
    try:

        parsed_answer = json.loads(
            raw_answer
        )

    except json.JSONDecodeError:

        parsed_answer = {
            "Correct Alternative": "",
            "Answer": "",
            "Justification": raw_answer,
            "Sources": []
        }

    # Valida e padroniza a resposta estruturada do LLM
    parsed_answer = validate_llm_answer(
        parsed_answer=parsed_answer,
        raw_answer=raw_answer
    )

    # Retorna todos os elementos da consulta
    return {

        "question": question.strip(),

        "correct_alternative": parsed_answer.get(
            "Correct Alternative",
            ""
        ),

        "answer": parsed_answer.get(
            "Answer",
            ""
        ),

        "justification": parsed_answer.get(
            "Justification",
            ""
        ),

        "sources": parsed_answer.get(
            "Sources",
            []
        ),

        "retrieved_chunks": retrieved_chunks,

        "raw_response": raw_answer,

        "prompt": full_prompt

    }

Salvamento dos Resultados

In [ ]:
# Salva cada pergunta em uma pasta própria.
# Cada nova execução da mesma pergunta gera um novo arquivo.
# Também atualiza automaticamente um arquivo de consistência.

def save_rag_result(
    rag_result,
    results_dir=RESULTS_DIR
):

    results_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    current_question = (
        rag_result["question"]
        .strip()
    )

    # --------------------------------------------------
    # 1. LOCALIZA OU CRIA A PASTA DA PERGUNTA
    # --------------------------------------------------

    question_dirs = sorted(
        [
            path
            for path in results_dir.iterdir()
            if path.is_dir()
            and re.fullmatch(
                r"\d+_pergunta_\d+",
                path.name
            )
        ]
    )

    question_dir = None
    question_number = None
    highest_question_number = 0

    for directory in question_dirs:

        match = re.fullmatch(
            r"(\d+)_pergunta_(\d+)",
            directory.name
        )

        if not match:
            continue

        folder_number = int(
            match.group(1)
        )

        highest_question_number = max(
            highest_question_number,
            folder_number
        )

        question_file = (
            directory
            / "pergunta.txt"
        )

        if question_file.exists():

            existing_question = (
                question_file
                .read_text(
                    encoding="utf-8"
                )
                .strip()
            )

            if existing_question == current_question:

                question_dir = directory
                question_number = folder_number

                break

    # Pergunta nova
    if question_dir is None:

        question_number = (
            highest_question_number + 1
        )

        question_dir = (
            results_dir
            / (
                f"{question_number:02d}_"
                f"pergunta_{question_number}"
            )
        )

        question_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        (
            question_dir
            / "pergunta.txt"
        ).write_text(
            current_question,
            encoding="utf-8"
        )

    # --------------------------------------------------
    # 2. DEFINE O NÚMERO DA NOVA EXECUÇÃO
    # --------------------------------------------------

    execution_files = [
        file_path
        for file_path in question_dir.glob(
            "execucao_*.txt"
        )
        if re.fullmatch(
            r"execucao_\d+\.txt",
            file_path.name
        )
    ]

    highest_execution_number = 0

    for file_path in execution_files:

        match = re.fullmatch(
            r"execucao_(\d+)\.txt",
            file_path.name
        )

        if match:

            highest_execution_number = max(
                highest_execution_number,
                int(
                    match.group(1)
                )
            )

    execution_number = (
        highest_execution_number + 1
    )

    output_file = (
        question_dir
        / f"execucao_{execution_number:02d}.txt"
    )

    # --------------------------------------------------
    # 3. MONTA O CONTEÚDO DA EXECUÇÃO
    # --------------------------------------------------

    lines = []

    lines.append(
        "=" * 80
    )

    lines.append(
        f"PERGUNTA Nº {question_number:02d} "
        f"- EXECUÇÃO Nº {execution_number:02d}"
    )

    lines.append(
        "=" * 80
    )

    lines.append("")

    lines.append(
        f"Data e hora: "
        f"{datetime.now().strftime('%d/%m/%Y %H:%M:%S')}"
    )

    lines.append(
        f"Modelo LLM: {LLM_MODEL_NAME}"
    )

    lines.append(
        f"Modelo de embeddings: {EMBEDDING_MODEL_NAME}"
    )

    lines.append(
        f"TOP_K: {TOP_K}"
    )

    lines.append("")

    # Pergunta
    lines.append(
        "PERGUNTA"
    )

    lines.append(
        "-" * 80
    )

    lines.append(
        current_question
    )

    lines.append("")

    # Alternativa
    lines.append(
        "ALTERNATIVA CORRETA"
    )

    lines.append(
        "-" * 80
    )

    lines.append(
        str(
            rag_result.get(
                "correct_alternative",
                ""
            )
        )
    )

    lines.append("")

    # Resposta
    lines.append(
        "RESPOSTA"
    )

    lines.append(
        "-" * 80
    )

    lines.append(
        str(
            rag_result.get(
                "answer",
                ""
            )
        )
    )

    lines.append("")

    # Justificativa
    lines.append(
        "JUSTIFICATIVA"
    )

    lines.append(
        "-" * 80
    )

    lines.append(
        str(
            rag_result.get(
                "justification",
                ""
            )
        )
    )

    lines.append("")

    # Fontes informadas pela LLM
    lines.append(
        "FONTES INFORMADAS PELO LLM"
    )

    lines.append(
        "-" * 80
    )

    sources = rag_result.get(
        "sources",
        []
    )

    if sources:

        for position, source in enumerate(
            sources,
            start=1
        ):

            lines.append(
                f"Fonte {position}"
            )

            lines.append(
                f"Artigo: "
                f"{source.get('Article', '')}"
            )

            lines.append(
                f"Chunk: "
                f"{source.get('Chunk', '')}"
            )

            lines.append("")

    else:

        lines.append(
            "Nenhuma fonte foi informada pelo modelo."
        )

        lines.append("")

    # --------------------------------------------------
    # CHUNKS RECUPERADOS PELO FAISS
    # --------------------------------------------------

    lines.append(
        "CHUNKS RECUPERADOS PELO FAISS"
    )

    lines.append(
        "=" * 80
    )

    lines.append("")

    for position, chunk in enumerate(
        rag_result.get(
            "retrieved_chunks",
            []
        ),
        start=1
    ):

        lines.append(
            f"RESULTADO {position}"
        )

        lines.append(
            "-" * 80
        )

        lines.append(
            f"Artigo: "
            f"{chunk.get('article_name', '')}"
        )

        lines.append(
            f"Autor: "
            f"{chunk.get('author', '')}"
        )

        lines.append(
            f"Ano: "
            f"{chunk.get('year', '')}"
        )

        lines.append(
            f"Título: "
            f"{chunk.get('title', '')}"
        )

        lines.append(
            f"Chunk: "
            f"{chunk.get('chunk_id', '')}"
        )

        lines.append(
            f"Similaridade: "
            f"{chunk.get('score', 0):.6f}"
        )

        lines.append(
            f"Fonte do embedding: "
            f"{chunk.get('embedding_source', '')}"
        )

        lines.append("")

        lines.append(
            "TEXTO USADO NA RECUPERAÇÃO VETORIAL"
        )

        lines.append(
            "-" * 80
        )

        lines.append(
            str(
                chunk.get(
                    "embedding_text",
                    ""
                )
            )
        )

        lines.append("")

        lines.append(
            "TEXTO ORIGINAL ENVIADO AO CONTEXTO"
        )

        lines.append(
            "-" * 80
        )

        lines.append(
            str(
                chunk.get(
                    "original_text",
                    ""
                )
            )
        )

        lines.append("")

        lines.append(
            "=" * 80
        )

        lines.append("")

    # Resposta bruta
    lines.append(
        "RESPOSTA BRUTA DO LLM"
    )

    lines.append(
        "-" * 80
    )

    lines.append(
        str(
            rag_result.get(
                "raw_response",
                ""
            )
        )
    )

    lines.append("")

    # Prompt completo
    lines.append(
        "PROMPT COMPLETO ENVIADO AO LLM"
    )

    lines.append(
        "-" * 80
    )

    lines.append(
        str(
            rag_result.get(
                "prompt",
                ""
            )
        )
    )

    lines.append("")

    lines.append(
        "=" * 80
    )

    # Salva a execução
    output_file.write_text(
        "\n".join(lines),
        encoding="utf-8"
    )

    # --------------------------------------------------
    # 4. LOCALIZA TODAS AS EXECUÇÕES APÓS O SALVAMENTO
    # --------------------------------------------------

    all_execution_files = [
        file_path
        for file_path in question_dir.glob(
            "execucao_*.txt"
        )
        if re.fullmatch(
            r"execucao_\d+\.txt",
            file_path.name
        )
    ]

    all_execution_files = sorted(
        all_execution_files,
        key=lambda file_path: int(
            re.fullmatch(
                r"execucao_(\d+)\.txt",
                file_path.name
            ).group(1)
        )
    )

    if not all_execution_files:

        raise RuntimeError(
            "Nenhum arquivo de execução foi encontrado "
            "para o cálculo da consistência."
        )

    # --------------------------------------------------
    # 5. LÊ AS ALTERNATIVAS DE TODAS AS EXECUÇÕES
    # --------------------------------------------------

    alternatives = []

    execution_results = []

    for file_path in all_execution_files:

        content = file_path.read_text(
            encoding="utf-8"
        )

        marker_start = (
            "ALTERNATIVA CORRETA\n"
            + "-" * 80
            + "\n"
        )

        marker_end = (
            "\n\nRESPOSTA"
        )

        alternative = ""

        if (
            marker_start in content
            and marker_end in content
        ):

            alternative = (
                content
                .split(
                    marker_start,
                    1
                )[1]
                .split(
                    marker_end,
                    1
                )[0]
                .strip()
                .upper()
            )

        if alternative in {
            "A", "B", "C", "D", "E"
        }:

            alternatives.append(
                alternative
            )

        execution_results.append(
            {
                "file": file_path.name,
                "alternative": alternative
            }
        )

    # --------------------------------------------------
    # 6. CALCULA A CONSISTÊNCIA
    # --------------------------------------------------

    most_common_alternative = ""
    consistency_percent = 0.0
    consistency_status = "Sem respostas válidas"

    if alternatives:

        counts = {
            alternative: alternatives.count(
                alternative
            )
            for alternative in {
                "A", "B", "C", "D", "E"
            }
        }

        max_count = max(
            counts.values()
        )

        predominant_alternatives = [
            alternative
            for alternative, count in counts.items()
            if count == max_count
            and count > 0
        ]

        consistency_percent = (
            max_count
            / len(alternatives)
            * 100
        )

        if len(
            predominant_alternatives
        ) == 1:

            most_common_alternative = (
                predominant_alternatives[0]
            )

            consistency_status = (
                "Alternativa predominante identificada"
            )

        else:

            most_common_alternative = (
                "EMPATE: "
                + ", ".join(
                    predominant_alternatives
                )
            )

            consistency_status = (
                "Empate entre alternativas"
            )

    # --------------------------------------------------
    # 7. SALVA O ARQUIVO DE CONSISTÊNCIA
    # --------------------------------------------------

    consistency_file = (
        question_dir
        / "consistencia.txt"
    )

    consistency_lines = []

    consistency_lines.append(
        "=" * 80
    )

    consistency_lines.append(
        f"CONSISTÊNCIA DA PERGUNTA Nº "
        f"{question_number:02d}"
    )

    consistency_lines.append(
        "=" * 80
    )

    consistency_lines.append("")

    consistency_lines.append(
        "PERGUNTA"
    )

    consistency_lines.append(
        "-" * 80
    )

    consistency_lines.append(
        current_question
    )

    consistency_lines.append("")

    consistency_lines.append(
        f"Total de execuções: "
        f"{len(all_execution_files)}"
    )

    consistency_lines.append(
        f"Execuções válidas: "
        f"{len(alternatives)}"
    )

    consistency_lines.append("")

    for item in execution_results:

        match = re.fullmatch(
            r"execucao_(\d+)\.txt",
            item["file"]
        )

        execution_id = (
            int(match.group(1))
            if match
            else 0
        )

        consistency_lines.append(
            f"Execução {execution_id:02d}: "
            f"{item['alternative']}"
        )

    consistency_lines.append("")

    consistency_lines.append(
        f"Alternativa predominante: "
        f"{most_common_alternative}"
    )

    consistency_lines.append(
        f"Consistência: "
        f"{consistency_percent:.2f}%"
    )

    consistency_lines.append(
        f"Status: "
        f"{consistency_status}"
    )

    consistency_lines.append("")

    consistency_lines.append(
        "=" * 80
    )

    consistency_file.write_text(
        "\n".join(
            consistency_lines
        ),
        encoding="utf-8"
    )

    # --------------------------------------------------
    # 8. RETORNA INFORMAÇÕES DO SALVAMENTO
    # --------------------------------------------------

    return {
        "file": output_file,
        "question_dir": question_dir,
        "question_number": question_number,
        "execution_number": execution_number,
        "consistency_file": consistency_file,
        "most_common_alternative": most_common_alternative,
        "consistency_percent": consistency_percent,
        "consistency_status": consistency_status,
        "total_executions": len(
            all_execution_files
        ),
        "valid_executions": len(
            alternatives
        )
    }

Verificação Final do Notebook

In [ ]:
# Realiza a verificação final do Notebook 04

print_header(
    "VERIFICAÇÃO FINAL DO NOTEBOOK 04"
)

checks = {

    "Índice FAISS": (
        index is not None
        and index.ntotal > 0
    ),

    "Metadados": (
        isinstance(metadata_records, list)
        and len(metadata_records) > 0
    ),

    "Modelo de embeddings": (
        embedding_model is not None
    ),

    "Cliente OpenRouter": (
        llm is not None
    ),

    "Função de recuperação": (
        callable(retrieve_chunks)
    ),

    "Construção do contexto": (
        callable(build_context)
    ),

    "Construção do prompt": (
        callable(build_prompt)
    ),

    "Consulta RAG": (
        callable(rag_query)
    ),

    "Salvamento por execução": (
        callable(save_rag_result)
    ),

    "Pasta de resultados": (
        RESULTS_DIR.exists()
        and RESULTS_DIR.is_dir()
    )
}

failed_checks = []

for name, status in checks.items():

    print_info(
        f"{name}:",
        "OK" if status else "ERRO"
    )

    if not status:

        failed_checks.append(
            name
        )

print()

print_info(
    "Vetores no índice:",
    index.ntotal
)

print_info(
    "Metadados:",
    len(metadata_records)
)

print_info(
    "Dimensão:",
    index_info["embedding_dimension"]
)

print_info(
    "Top-K:",
    TOP_K
)

print_info(
    "Modelo de embeddings:",
    EMBEDDING_MODEL_NAME
)

print_info(
    "Modelo LLM:",
    LLM_MODEL_NAME
)

print_info(
    "Temperatura:",
    "Padrão do modelo/provedor (não definida manualmente)"
)

print()

# Recupera as contagens das fontes dos embeddings

source_counts = index_info.get(
    "embedding_source_counts",
    {}
)

print_info(
    "Fonte cleaned_summary:",
    source_counts.get(
        "cleaned_summary",
        0
    )
)

print_info(
    "Fonte summary:",
    source_counts.get(
        "summary",
        0
    )
)

print_info(
    "Fonte text:",
    source_counts.get(
        "text",
        0
    )
)

print()

print_info(
    "Pasta de resultados:",
    RESULTS_DIR
)

print_info(
    "Estrutura de salvamento:",
    "Pasta por pergunta + arquivo por execução"
)

print_info(
    "Arquivo de consistência:",
    "Atualizado automaticamente"
)

# Valida consistência entre índice e metadados

if index.ntotal != len(
    metadata_records
):

    raise ValueError(
        "O número de vetores do índice não corresponde "
        "ao número de metadados."
    )

# Valida dimensão

if index_info["embedding_dimension"] != 768:

    raise ValueError(
        "A dimensão dos embeddings está incorreta."
    )

# Valida total das fontes

if sum(
    source_counts.values()
) != len(
    metadata_records
):

    raise ValueError(
        "A contagem das fontes dos embeddings "
        "não corresponde ao total de metadados."
    )

if failed_checks:

    print()

    print_error(
        "Existem componentes que precisam ser revisados."
    )

    print()

    print(
        "Componentes com problema:"
    )

    for item in failed_checks:

        print(
            f"• {item}"
        )

    raise RuntimeError(
        "A verificação final do Notebook 04 encontrou erros."
    )

print_success(
    "Notebook 04 configurado e validado com sucesso."
)

print("=" * 70)


 VERIFICAÇÃO FINAL DO NOTEBOOK 04
Índice FAISS:             OK
Metadados:                OK
Modelo de embeddings:     OK
Cliente OpenRouter:       OK
Função de recuperação:    OK
Construção do contexto:   OK
Construção do prompt:     OK
Consulta RAG:             OK
Salvamento por execução:  OK
Pasta de resultados:      OK

Vetores no índice:        6844
Metadados:                6844
Dimensão:                 768
Top-K:                    3
Modelo de embeddings:     sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Modelo LLM:               meta-llama/llama-3.1-8b-instruct
Temperatura:              Padrão do modelo/provedor (não definida manualmente)

Fonte cleaned_summary:    6679
Fonte summary:            7
Fonte text:               158

Pasta de resultados:      /content/drive/MyDrive/RAG_Novo/04_Resultados
Estrutura de salvamento:  Pasta por pergunta + arquivo por execução
Arquivo de consistência:  Atualizado automaticamente

✅ Notebook 04 configurado e validado com suce

Interface de Consulta

In [ ]:
# Interface automática para executar o questionário completo no RAG
# Cada questão é enviada em uma chamada independente à LLM.
# A sequência Q1 -> Q20 é repetida pelo número de rodadas definido.

# ============================================================
# CONFIGURAÇÃO DO EXPERIMENTO
# ============================================================

NUM_RODADAS = 10

# Para alterar futuramente:
# NUM_RODADAS = 20
# NUM_RODADAS = 30
# etc.


# ============================================================
# QUESTIONÁRIO
# ============================================================

QUESTIONS = [

    """
Technology Roadmapping started in companies, not in the academy.
What company was a pioneer in roadmapping and is mentioned in an academic paper published in the 80's?
A) Philips
B) Motorola
C) General Motors
D) Lucent Technologies
E) Nasa
""",

    """
The roadmap layers can be organized to support diffent sectors.
What have been the most used roadmap layers, starting from the top to the bottom, proposed in the academy
to create a fast-start roadmapping approach?
A) Innovation, Risk, Value
B) Technology, Market, Business
C) Knowledge, Culture, Strategy
D) Market, Product, Technology
E) Finance, Operations, Sales
""",

    """
What is the core motivation to apply the roadmapping approach?
A) Benchmark competitors working with similar technologies
B) Develop a strategic plan for products and technologies
C) Develop a business plan focused on technological innovation
D) Replace project management tools with updated gant charts and product roadmaps
E) Map the business trajectory considering different scenarios
""",

    """
What factors can be considered the most important when defining the roadmap timeline:
A) Industrial sector and business strategy
B) Market share and dynamics
C) Government regulation and intellectual property
D) Company and market size
E) Leadership and cultural approach
""",

    """
Roadmapping processes can provide tangible and intangible outcomes.
What is the main tangible output of a roadmapping process:
A) Business case integrated with a technological plan
B) Product plan communication in different departments
C) Strategic roadmap describing product and technological goals
D) Alignment and consensus among the roadmapping stakeholders
E) Sharing of knowledge developed through roadmapping
""",

    """
What are the main two approaches used in practice to support roadmapping applications?
A) Expert and data-driven roadmapping
B) Expert and project-based roadmapping
C) Strategic and product-based roadmapping
D) Computer-based and data-driven roadmapping
E) Vectorial and tabular-based roadmapping
""",

    """
What are the two innovation and technology tools indicated in the alternatives that can help with risk mitigation in roadmapping?
A) Quality function deployment and functional modeling
B) Scenario planning and portfolio management
C) Linking grids and morphological matrices
D) SWOT and Porter's 5 forces
E) Technology readiness level (TRL) and gantt charts
""",

    """
What alternative is NOT a barrier to the application of roadmapping?
A) Lack of strategic information during the early stages
B) Fast-changing customer requirements related to uncertain environments
C) Uncertainties in technology maturity and ecosystems
D) Application of product features and platforms as strategic orientation
E) Dynamic markets and competition when applied to small and medium companies
""",

    """
The expert-based roadmapping has been the most used approach over the years.
What technique does it adopt as central for roadmapping development?
A) Market analysis and trend modeling support by experts
B) Computational forecasting integrated with topic modeling
C) Collaborative and multidisciplinary workshops
D) Long-term projects created by specialized engineering teams
E) Teams focused on the technologies required for the roadmap
""",

    """
Several research institutes have supported roadmapping over time, forming the roadmapping schools of thought.
What is the academic institution most acknowledged for its contribution to roadmapping, with influence on all other schools?
A) Portland State University
B) University of Sao Paulo
C) University of Cambridge
D) National Seoul University
E) Northwestern University
""",

    """
Roadmapping and roadmaps refers to different parts of a roadmapping approach.
A roadmap can be best defined as:
A) A map that presents the entire story of business development
B) A guide to support strategic decisions in dynamic and uncertain technological domains
C) A chart that describes the actions planned for the company in the long-term.
D) A vision of the goals to be achieved by a business developed by a group of experts.
E) A layered and temporal graphical representation of strategic options to a company pursue
regarding innovation and technology development.
""",

    """
Despite some resistance, recent studies started using digital technologies for roadmapping.
What alternative present advantages noted in a digital roadmapping approach:
A) Better management of technology uncertainty and use of digital communication
B) Facilitated collaboration from different locations and faster information processing
C) Development of multiple roadmapping types and use of roadmap software
D) Ensure better engagement of experts in the process because of conference meetings and video sharing.
E) Support better collection of information for the technological layer and improved experts' discussion.
""",

    """
The application of scenario planning integrated with roadmapping is a common practice to improve roadmapping results.
What alternative indicates a benefit that scenario planning brings to roadmapping:
A) It helps to better address the long-term and vision strategies
B) It helps to better connect products and technologies' interdependencies
C) It helps to reduce the lead time to reach a qualified roadmap result
D) It helps to organize the roadmapping workshops and clarify the expected results in uncertain markets
E) It helps to assess the maturity of the existing technologies in multiple marks.
""",

    """
The development of an expert-based roadmapping involves multidisciplinary teams that need support to reach the expected result.
What is the alternative that presents a relevant role in this type of roadmapping to ensure collaborative work?
A) Workshop facilitator
B) Technical consultant
C) Business manager
D) Engineering expert
E) Marketing expert
""",

    """
There are several quantitative techniques used for computer or data-driven roadmapping.
These techniques have used data from several sources, but the main source used for this type of roadmapping is:
A) Social networks data
B) Patent data
C) Industrial reports data
D) Marketing data
E) Quantitative optimization data
""",

    """
What is NOT suggested as a practice to improve roadmapping performance and results?
A) Development of a pilot roadmapping project in companies with little experience
B) Define clear roadmapping goals and outcome expectations
C) Define a clear and feasible unit of analysis embracing the main product features
D) The involvement of experts with a technical and commercial background
E) Address roadmapping as a one-time application, without considering relevant values for the implementation of its result
""",

    """
The connection between roadmap layers is a relevant practice, in particular for product-technology roadmaps.
What tool is especially relevant for connecting roadmapping layers?
A) Linking grids
B) Technology Re-Linking Levels (TRL)
C) STEEP and SWOT
D) Scenario Planning
E) Decision Criteria
""",

    """
The roadmap layout is built upon a structure designed to organize information in layers and a timeline.
What are the three questions used to guide the collection of information that needs to be organized through the roadmap layers?
A) Where are we now? Where do we want to go? How do we get there?
B) Why do we need it? What do we need? How do we develop it?
C) Who are our customers? What do they buy? How do we contact them?
D) What are our strategies? How do we implement i? How do we measure results?
E) What is our vision? What is our current situation? What are our plans?
""",

    """
The roadmapping approach can be defined as:
A) A management approach used to map and manage product projects successfully launched in the market
B) A tool that supports improved decision-making based on business strategies and marketing goals
C) A management tool used to support technical departments in deciding what technology to develop
D) As approach that aims to develop a roadmap containing business directions prioritized for different market segments
E) A management approach used to identify, define, and map innovation strategies, objectives, and actions within a business or organization
""",

    """
The main outcome of roadmapping, the roadmap, can be also used as a diagnostic tool.
How the roadmap visual features can be used to guide roadmapping improvements:
A) They can indicate areas in which the roadmapping process is missing information or provides insufficient results
B) They can report with figures if the roadmapping process was conducted with commitment
C) They can show if the roadmapping experts were chosen correctly to provide the required information
D) They can present visually the business priorities that need to be considered in strategic actions
E) They can use symbols and signals to help communicate results and support organizational adoption
"""
]


# ============================================================
# VALIDAÇÕES INICIAIS
# ============================================================

if NUM_RODADAS <= 0:

    raise ValueError(
        "NUM_RODADAS deve ser maior que zero."
    )

if len(QUESTIONS) != 20:

    raise ValueError(
        f"O questionário deveria conter 20 questões, "
        f"mas contém {len(QUESTIONS)}."
    )


# ============================================================
# INÍCIO DO EXPERIMENTO
# ============================================================

total_questions = len(QUESTIONS)

total_expected_calls = (
    total_questions
    * NUM_RODADAS
)

successful_calls = 0
failed_calls = 0

failed_executions = []


print_header(
    "EXECUÇÃO AUTOMÁTICA DO QUESTIONÁRIO RAG"
)

print_info(
    "Número de questões:",
    total_questions
)

print_info(
    "Número de rodadas:",
    NUM_RODADAS
)

print_info(
    "Total previsto de chamadas:",
    total_expected_calls
)

print_info(
    "Modelo LLM:",
    LLM_MODEL_NAME
)

print_info(
    "Temperatura:",
    "Padrão do modelo/provedor (não definida manualmente)"
)

print_info(
    "Top-K:",
    TOP_K
)

print()

print_warning(
    "Cada questão será enviada em uma chamada independente à LLM."
)

print("=" * 70)


# ============================================================
# EXECUÇÃO DAS RODADAS
# ============================================================

for round_number in range(
    1,
    NUM_RODADAS + 1
):

    print()

    print_header(
        f"RODADA {round_number:02d} DE {NUM_RODADAS:02d}"
    )

    for question_number, question in enumerate(
        QUESTIONS,
        start=1
    ):

        print()

        print(
            f"[Rodada {round_number:02d}/{NUM_RODADAS:02d}] "
            f"Processando questão "
            f"{question_number:02d}/{total_questions:02d}..."
        )

        try:

            # ----------------------------------------------
            # CHAMADA INDEPENDENTE AO RAG
            # ----------------------------------------------

            result = rag_query(
                question=question.strip(),
                top_k=TOP_K
            )

            # ----------------------------------------------
            # SALVAMENTO INDIVIDUAL DA EXECUÇÃO
            # ----------------------------------------------

            save_info = save_rag_result(
                result
            )

            successful_calls += 1

            print_info(
                "Alternativa:",
                (
                    result["correct_alternative"]
                    if result["correct_alternative"]
                    else "Não identificada"
                )
            )

            print_info(
                "Arquivo:",
                save_info["file"].name
            )

            print_info(
                "Consistência atual:",
                f"{save_info['consistency_percent']:.2f}%"
            )

            print_success(
                f"Questão {question_number:02d} concluída e salva."
            )

        except Exception as error:

            failed_calls += 1

            failed_executions.append(
                {
                    "round": round_number,
                    "question": question_number,
                    "error": str(error)
                }
            )

            print_error(
                f"Erro na questão {question_number:02d} "
                f"da rodada {round_number:02d}."
            )

            print(
                str(error)
            )

            print_warning(
                "A execução continuará para a próxima questão."
            )


# ============================================================
# RESUMO FINAL
# ============================================================

print()

print_header(
    "EXPERIMENTO CONCLUÍDO"
)

print_info(
    "Rodadas solicitadas:",
    NUM_RODADAS
)

print_info(
    "Questões por rodada:",
    total_questions
)

print_info(
    "Chamadas previstas:",
    total_expected_calls
)

print_info(
    "Chamadas concluídas:",
    successful_calls
)

print_info(
    "Chamadas com erro:",
    failed_calls
)

print_info(
    "Pasta de resultados:",
    RESULTS_DIR
)

print()

if failed_calls == 0:

    print_success(
        "Todas as chamadas foram concluídas e salvas com sucesso."
    )

else:

    print_warning(
        "O experimento terminou, mas algumas chamadas apresentaram erro."
    )

    print()

    print(
        "Execuções com erro:"
    )

    for failure in failed_executions:

        print(
            f"• Rodada {failure['round']:02d} | "
            f"Questão {failure['question']:02d} | "
            f"{failure['error']}"
        )

print("=" * 70)


 EXECUÇÃO AUTOMÁTICA DO QUESTIONÁRIO RAG
Número de questões:       20
Número de rodadas:        10
Total previsto de chamadas: 200
Modelo LLM:               meta-llama/llama-3.1-8b-instruct
Temperatura:              Padrão do modelo/provedor (não definida manualmente)
Top-K:                    3


⚠️ Cada questão será enviada em uma chamada independente à LLM.


 RODADA 01 DE 10

[Rodada 01/10] Processando questão 01/20...
Alternativa:              B
Arquivo:                  execucao_01.txt
Consistência atual:       100.00%

✅ Questão 01 concluída e salva.

[Rodada 01/10] Processando questão 02/20...
Alternativa:              D
Arquivo:                  execucao_01.txt
Consistência atual:       100.00%

✅ Questão 02 concluída e salva.

[Rodada 01/10] Processando questão 03/20...
Alternativa:              Não identificada
Arquivo:                  execucao_01.txt
Consistência atual:       0.00%

✅ Questão 03 concluída e salva.

[Rodada 01/10] Processando questão 04/20...
Alternativa: 

Resumo da Arquitetura do Sistema RAG

O sistema desenvolvido utiliza uma arquitetura de **Retrieval-Augmented Generation (RAG)** para responder questões de múltipla escolha relacionadas a *Technology and Innovation Roadmapping*.

O fluxo integra recuperação semântica de informações e geração de respostas por um modelo de linguagem. A pergunta do usuário é transformada em embedding utilizando o mesmo modelo empregado na indexação da base documental, o **`sentence-transformers/paraphrase-multilingual-mpnet-base-v2`**.

Os embeddings da base documental foram gerados a partir de uma estratégia de prioridade textual definida no Notebook 02: **`cleaned_summary → summary → text`**. Assim, quando o campo `cleaned_summary` está disponível, ele é utilizado; na sua ausência, utiliza-se o campo `summary` e, caso ambos estejam vazios, o texto original (`text`) é utilizado como fallback. A origem textual utilizada para cada embedding é registrada no campo **`embedding_source`**.

Em seguida, o índice vetorial **FAISS**, configurado como `IndexFlatIP`, realiza a busca semântica e recupera os três chunks mais relevantes para a pergunta (`TOP_K = 3`). A recuperação é realizada a partir dos embeddings previamente indexados e dos valores de similaridade calculados entre o embedding da pergunta e os embeddings da base documental.

Após a recuperação, os **textos originais (`original_text`)** associados aos chunks selecionados são utilizados para construir o contexto enviado ao modelo de linguagem. Dessa forma, o conteúdo utilizado para a recuperação vetorial pode ter origem em `cleaned_summary`, `summary` ou `text`, enquanto a etapa de geração da resposta utiliza o texto original dos chunks recuperados.

O contexto recuperado, juntamente com a pergunta e suas alternativas, compõe o prompt enviado ao **Llama 3.1 8B Instruct**, acessado por meio da API do **OpenRouter**. O modelo compara as alternativas com as informações presentes no contexto recuperado e retorna uma resposta estruturada contendo a alternativa selecionada, o texto da resposta, uma justificativa e as principais fontes utilizadas.

Por fim, o sistema apresenta a resposta ao usuário e registra automaticamente os detalhes de cada consulta na pasta de resultados. O arquivo gerado inclui a pergunta, a resposta produzida pelo modelo, a justificativa, as fontes informadas pela LLM, os chunks efetivamente recuperados pelo FAISS, os valores de similaridade, a origem do embedding (`embedding_source`), o texto utilizado na recuperação vetorial, o texto original enviado ao contexto, a resposta bruta da LLM e o prompt completo utilizado na consulta.

Fluxo do Sistema RAG

Pergunta do usuário (A–E)  
↓  
Modelo de Embeddings  
`sentence-transformers/paraphrase-multilingual-mpnet-base-v2`  
↓  
Embedding da pergunta (768 dimensões)  
↓  
Índice FAISS (`IndexFlatIP`)  
*Base indexada a partir de `cleaned_summary → summary → text`*  
↓  
Recuperação semântica (`TOP_K = 3`)  
↓  
Top-3 chunks + metadados + `embedding_source`  
↓  
Recuperação dos textos originais (`original_text`)  
↓  
Construção do contexto  
↓  
Construção do prompt RAG  
↓  
OpenRouter  
↓  
Llama 3.1 8B Instruct  
↓  
Resposta estruturada  
*Alternativa + Resposta + Justificativa + Fontes*  
↓  
Interface de Consulta  
↓  
Salvamento automático em `04_Resultados`